In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.competitions_data"

def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(f"`{column}`")) == "",
                    None
                ).otherwise(
                    F.trim(F.col(f"`{column}`"))
                )
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze table
# ---------------------------------------------------------

table_name = f"{BRONZE_SCHEMA}.api_competitions_data"

if not spark.catalog.tableExists(table_name):
    print(f"WARNING: Table does not exist: {table_name}")

df = spark.table(table_name)
df = clean_strings(df)

# ---------------------------------------------------------
# 2. Select and standardize columns
# ---------------------------------------------------------

df_silver = (
    df
    .select(
        F.explode("response").alias("response"),
        F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
        F.col("_source_file").cast("string").alias("_bronze_source_file"),
        F.col("_source_system").cast("string").alias("_source_system")
    )
    .select(
        F.col("response.league.id").alias("league_id"),
        F.col("response.league.name").alias("league_name"),
        F.col("response.league.type").alias("league_type"),
        F.col("response.country.code").alias("country_code"),
        F.col("response.country.name").alias("country_name"),

        # Bronze metadata
        F.col("_bronze_ingested_at"),
        F.col("_bronze_source_file"),
        F.col("_source_system")
    )
)

# ---------------------------------------------------------
# 3. Remove duplicates
# ---------------------------------------------------------

duplicate_columns = [
    "league_id"
]

duplicate_count = (
    df_silver
    .groupBy(duplicate_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups found: {duplicate_count}")

if duplicate_count > 0:
    df_silver = df_silver.dropDuplicates(duplicate_columns)
    print("Duplicates removed.")
else:
    print("No duplicates found. No changes made.")


# ---------------------------------------------------------
# 4. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 5. Basic validation
# ---------------------------------------------------------

# Null league IDs
null_league_ids = df_silver.filter(
    F.col("league_id").isNull()
).count()

if null_league_ids > 0:
    print(f"WARNING: NULL league_id: {null_league_ids}")


# Null league names
null_league_names = df_silver.filter(
    F.col("league_name").isNull()
).count()

if null_league_names > 0:
    print(f"WARNING: NULL league_name: {null_league_names}")
    
print(f"Silver rows: {df_silver.count()}")

# ---------------------------------------------------------
# 6. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")